# Incrementally ingest the CSV batches

- `batch_id` selects the simulated CRM customer/sales batch.
- Product and ERP files are seeded into their landing folders only if missing.
- This makes the streaming job independent from the batch job.


In [ ]:
dbutils.widgets.text("batch_id", "1")

batch_id = int(dbutils.widgets.get("batch_id"))

if batch_id < 1:
    raise ValueError("batch_id must be >= 1")

print(f"Releasing simulated batch {batch_id}")


In [ ]:
BASE = (
    "/Volumes/data_lakehouse_databricks/"
    "bronze/landing_vol"
)

BRONZE_SOURCE_BASE = (
    "/Volumes/data_lakehouse_databricks/"
    "bronze/bronze_vol"
)

sales_source = (
    f"{BASE}/generated_sales/batch_{batch_id}"
)

customers_source = (
    f"{BASE}/generated_cust/batch_{batch_id}"
)

sales_target = (
    f"{BASE}/landing_sales/"
    f"sales_{batch_id:03d}.csv"
)

customers_target = (
    f"{BASE}/landing_cust/"
    f"cust_{batch_id:03d}.csv"
)

STATIC_FILES = {
    "product": {
        "source": (
            f"{BRONZE_SOURCE_BASE}/"
            "source_crm/prd_info.csv"
        ),
        "target": (
            f"{BASE}/landing_product/"
            "prd_info.csv"
        ),
    },
    "customer_demographics": {
        "source": (
            f"{BRONZE_SOURCE_BASE}/"
            "source_erp/cust_az12.csv"
        ),
        "target": (
            f"{BASE}/"
            "landing_customer_demographics/"
            "cust_az12.csv"
        ),
    },
    "customer_location": {
        "source": (
            f"{BRONZE_SOURCE_BASE}/"
            "source_erp/loc_a101.csv"
        ),
        "target": (
            f"{BASE}/"
            "landing_customer_location/"
            "loc_a101.csv"
        ),
    },
    "product_category": {
        "source": (
            f"{BRONZE_SOURCE_BASE}/"
            "source_erp/px_cat_g1v2.csv"
        ),
        "target": (
            f"{BASE}/"
            "landing_product_category/"
            "px_cat_g1v2.csv"
        ),
    },
}


## Helper functions

In [ ]:
def path_exists(path):
    parent, name = path.rsplit("/", 1)

    try:
        return any(
            file_info.name == name
            for file_info in dbutils.fs.ls(parent)
        )
    except Exception:
        return False


def copy_single_csv_from_directory(
    source_directory,
    target_file,
):
    csv_files = [
        file_info
        for file_info in dbutils.fs.ls(source_directory)
        if file_info.name.lower().endswith(".csv")
    ]

    if len(csv_files) != 1:
        raise RuntimeError(
            f"Expected exactly one CSV in "
            f"{source_directory}, found "
            f"{len(csv_files)}."
        )

    parent = target_file.rsplit("/", 1)[0]
    dbutils.fs.mkdirs(parent)

    dbutils.fs.cp(
        csv_files[0].path,
        target_file,
    )


def ensure_static_file(
    source_file,
    target_file,
):
    parent = target_file.rsplit("/", 1)[0]
    dbutils.fs.mkdirs(parent)

    if path_exists(target_file):
        print(
            f"Static file already present: "
            f"{target_file}"
        )
        return

    dbutils.fs.cp(
        source_file,
        target_file,
    )

    print(
        f"Initialized static source: "
        f"{target_file}"
    )


## Sales

In [ ]:
if path_exists(sales_target):
    raise ValueError(
        f"Batch {batch_id} was already "
        f"released for sales: "
        f"{sales_target}"
    )

copy_single_csv_from_directory(
    sales_source,
    sales_target,
)

print(
    f"Released sales batch {batch_id}: "
    f"{sales_target}"
)


## Customers

In [ ]:
if path_exists(customers_target):
    raise ValueError(
        f"Batch {batch_id} was already "
        f"released for customers: "
        f"{customers_target}"
    )

copy_single_csv_from_directory(
    customers_source,
    customers_target,
)

print(
    f"Released customer batch {batch_id}: "
    f"{customers_target}"
)


## Static CRM / ERP source tables

These are copied only once. Later runs leave them untouched.


In [ ]:
for source_name, config in STATIC_FILES.items():
    ensure_static_file(
        source_file=config["source"],
        target_file=config["target"],
    )

print("Landing-area preparation completed.")
